# Exercise 3

Get familiar with the API documentation [boredapi](https://bored-api.appbrewery.com/). This is an API that suggests a random activity you can engage in to kill boredom. Find in it how to filter the returned activities based on the number of participants.

Using the `input(...)` function, ask the user how many friends they want to meet.

With the `requests` library query the API (remember about the `participants` filter), and display the suggested activity.

Remember that when giving the number of people in the query, in addition to the number of friends, you need to also include the user of the script! 

In [13]:
import json
import requests

number_of_participants = int(input("Enter number of participants (Including You):\t"))
url = f'https://bored-api.appbrewery.com/filter?participants={number_of_participants}'

response = requests.get(url)

data = response.json()
print(data)

[{'activity': 'Learn Express.js', 'availability': 0.25, 'type': 'education', 'participants': 1, 'price': 0.1, 'accessibility': 'Few to no challenges', 'duration': 'hours', 'kidFriendly': True, 'link': 'https://expressjs.com/', 'key': '3943506'}, {'activity': 'Learn to greet someone in a new language', 'availability': 0.2, 'type': 'education', 'participants': 1, 'price': 0.1, 'accessibility': 'Few to no challenges', 'duration': 'minutes', 'kidFriendly': True, 'link': '', 'key': '4704256'}, {'activity': 'Learn how to play a new sport', 'availability': 0.2, 'type': 'recreational', 'participants': 1, 'price': 0.1, 'accessibility': 'Minor challenges', 'duration': 'minutes', 'kidFriendly': True, 'link': '', 'key': '5808228'}, {'activity': 'Learn a new programming language', 'availability': 0.25, 'type': 'education', 'participants': 1, 'price': 0.1, 'accessibility': 'Few to no challenges', 'duration': 'hours', 'kidFriendly': True, 'link': '', 'key': '5881028'}, {'activity': 'Learn how to fold

In [14]:
import time

def get_activity(participants, max_retries=3):
    url = f'https://bored-api.appbrewery.com/filter?participants={participants}'
    backoff = 1
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 429:
                # Rate limited by the API - respect Retry-After if provided
                retry_after = resp.headers.get('Retry-After')
                wait = int(retry_after) if retry_after and retry_after.isdigit() else backoff
                print(f'Rate limited (429). Waiting {wait} seconds before retrying...')
                time.sleep(wait)
                backoff *= 2
                continue
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException as e:
            print(f'Request error (attempt {attempt}):', e)
            if attempt < max_retries:
                time.sleep(backoff)
                backoff *= 2
            else:
                raise

# Validate input
while True:
    try:
        number_of_participants = int(input("Enter number of participants (Including You):	"))
        if number_of_participants <= 0:
            print('Please enter a positive integer.')
            continue
        break
    except ValueError:
        print('Please enter a valid integer.')

try:
    data = get_activity(number_of_participants)
except Exception as e:
    print('Failed to retrieve activity:', e)
else:
    if not data:
        print('No activity found for', number_of_participants)
    else:
        # The 'filter' endpoint typically returns a list of activities
        activity = None
        if isinstance(data, list):
            activity = data[0] if data else None
        elif isinstance(data, dict):
            activity = data
        if not activity:
            print('No activity suggested.')
        else:
            print('Suggested activity:')
            print(' -', activity.get('activity'))
            print(' - Type:', activity.get('type'))
            print(' - Participants:', activity.get('participants'))
            if 'price' in activity:
                print(' - Price:', activity.get('price'))

Please enter a valid integer.
Please enter a positive integer.
Suggested activity:
 - Learn Express.js
 - Type: education
 - Participants: 1
 - Price: 0.1
